In [ ]:
import cv2
import glob
import matplotlib.pyplot as plt
import os
import numpy as np

dir = '20220727/*/*' # Directory containing the images

# Load the images along with their file names
images = [(cv2.imread(file), file) for file in glob.glob(dir)]

# Convert images to grayscale
gray_images = [(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), file) for img, file in images]

# Apply thresholding
threshold = 200  # Adjust this value as needed
binary_images = [(cv2.threshold(gray_img, threshold, 255, cv2.THRESH_BINARY)[1], file) for gray_img, file in gray_images]

# Find the first image with significant sky
first_sky_image_index = None
first_sky_image_file = None
for i, (binary_img, file) in enumerate(binary_images):
    plt.imshow(images[i][0])
    
    sky_pixels = cv2.countNonZero(binary_img)
    if sky_pixels > threshold:
        first_sky_image_index = i
        first_sky_image_file = file
        if "_sky" not in file:
            new_file_name = os.path.splitext(file)[0] + "_sky" + os.path.splitext(file)[1]
            os.rename(file, new_file_name)
    else:
        if "_allcloud" not in file:
            new_file_name = os.path.splitext(file)[0] + "_allcloud" + os.path.splitext(file)[1]
            os.rename(file, new_file_name)
        


In [ ]:
root_dir = '20220727/'

# Loop through all the directories in root_dir
for dir_name, sub_dirs, files in os.walk(root_dir):
    if dir_name == root_dir:
        continue    
    # Check if the directory contains only files with the "_allcloud" tag
    allcloud_files = glob.glob(os.path.join(dir_name, "*_allcloud.*"))
    if len(files) == len(allcloud_files):
        print("Directory containing only '_allcloud' files:", dir_name)
        if "_allcloud" not in dir_name:
            new_dir_name = dir_name + "_allcloud"
            os.rename(dir_name, new_dir_name)

In [ ]:
import cv2
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import os
# Load the image
image = cv2.imread('20220727/pass_13_181947_rfc/frame_c305_20220727_181950_sky.png')

# Convert the image from BGR to HSV color space
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# Define range for blue color in HSV
lower_blue = np.array([110,50,50])
upper_blue = np.array([130,255,255])

# Threshold the HSV image to get only blue colors
mask = cv2.inRange(hsv, lower_blue, upper_blue)

# Bitwise-AND mask and original image
res = cv2.bitwise_and(image, image, mask=mask)

# Display the original image and the result
plt.imshow(image)
plt.imshow(res)

# Count the number of blue pixels
blue_pixels = cv2.countNonZero(mask)

print("Number of blue pixels:", blue_pixels)

In [ ]:
import cv2
import numpy as np
import glob
import os

dir = '20220727/*/*_sky.*' # Directory containing the images

# Load the images along with their file names
images = [(cv2.imread(file), file) for file in glob.glob(dir)]

for img, file in images:
    # Convert the image from BGR to HSV color space
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Define range for blue color in HSV
    lower_blue = np.array([110,50,50])
    upper_blue = np.array([130,255,255])

    # Threshold the HSV image to get only blue colors
    mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Count the number of blue pixels
    blue_pixels = cv2.countNonZero(mask)

    # If the image has more than 20000 blue pixels, rename it to include the "_bluesky" tag
    if blue_pixels > 10000:
        if "_bluesky" not in file:
            new_file_name = os.path.splitext(file)[0] + "_bluesky" + os.path.splitext(file)[1]
            os.rename(file, new_file_name)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import glob
import cv2
import re
import os

root_dir = '20220727/'

# Define the key function to extract the date and timestamp from the file name
def get_datetime_from_filename(file):
    match = re.search(r'frame_c305_(\d{8})_(\d{6})', file)
    if match:
        date = match.group(1)
        
        timestamp = match.group(2)
        
        return int(date + timestamp)
    else:
        print("No match")
        return 0



# Loop through all the directories in root_dir
for dir_name, sub_dirs, files in os.walk(root_dir):
    subdir_name = os.path.basename(dir_name)
    print(subdir_name)
    # Get all the images in the directory
    image_files = glob.glob(os.path.join(dir_name, "*.png"))  # Adjust the file extension as needed
 
    image_files.sort(key=lambda f: re.search(r'(\d{6})_\w+', f).group(1) if re.search(r'(\d{6})_\w+', f) else '')
    # Sort the image files based on the extracted datetime
    image_files = sorted(image_files, key=get_datetime_from_filename)
   
    # Create a new figure for the directory
    fig = plt.figure(figsize=(10, 10))

    # Calculate the number of rows and columns based on the number of images
    num_images = len(image_files)
    num_rows = (num_images + 4) // 5  # Adjust the grid size as needed
    num_cols = min(num_images, 5)
    
    
    # Loop through the images and add them to the figure
    for i, file in enumerate(image_files):
        img = mpimg.imread(file)
        
        
        
        # If the image name includes the tag "_bluesky", add a green border to the image
        if "_bluesky" in file:
            border_color = (0, 255, 0)  # Green color
            border_width = 20
            img = cv2.copyMakeBorder(img, border_width, border_width, border_width, border_width, cv2.BORDER_CONSTANT, value=border_color)
            #img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ax = fig.add_subplot(num_rows, num_cols, i+1)
        ax.imshow(img)
        ax.axis('off') # save figure
    if num_images > 0:
        plt.savefig(root_dir+'/' + subdir_name + '.png')
        plt.close(fig)
        break
    

    

In [ ]:
print(root_dir+'/' + subdir_name + '.png')